Switch to a dataset that feels more like time-series data; this will match the code in the E-certification textbook

In [1]:
# Install if needed (skip if already installed)
# %pip install -q datasets torch

import re
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


# ----------------------------
# Data loading (Hugging Face datasets)
# ----------------------------
ds = load_dataset("ag_news")  # ds["train"], ds["test"]

class AGNewsHFDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # HF ag_news labels are 0..3 (unlike torchtext which uses 1..4)
        return int(item["label"]), item["text"]


train_ds = AGNewsHFDataset(ds["train"])
test_ds  = AGNewsHFDataset(ds["test"])


# ----------------------------
# Tokenizer (simple basic_english equivalent without torchtext)
# ----------------------------
def basic_english_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # replace symbols with spaces
    text = re.sub(r"\s+", " ", text).strip()    # collapse whitespace
    return text.split()


# ----------------------------
# Vocabulary (built without torchtext)
# ----------------------------
class Vocab:
    def __init__(self, stoi, itos, default_index: int):
        self.stoi = stoi
        self.itos = itos
        self.default_index = default_index

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token: str):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self[token] for token in tokens]


def build_vocab_from_texts(text_iter, tokenizer, specials=("<unk>", "<pad>"), min_freq=2):
    counter = Counter()
    for text in text_iter:
        toks = tokenizer(text)
        if not toks:
            toks = ["<unk>"]
        counter.update(toks)

    itos = list(specials)
    # sort by freq desc, then token asc for stability
    for tok, freq in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        if freq >= min_freq and tok not in specials:
            itos.append(tok)

    stoi = {tok: i for i, tok in enumerate(itos)}
    default_index = stoi["<unk>"]
    return Vocab(stoi=stoi, itos=itos, default_index=default_index)


# Build vocab from training set
vocab = build_vocab_from_texts(
    (ds["train"][i]["text"] for i in range(len(ds["train"]))),
    tokenizer=basic_english_tokenizer,
    specials=("<unk>", "<pad>"),
    min_freq=2
)
pad_idx = vocab["<pad>"]


def text_pipeline(x: str):
    tokens = basic_english_tokenizer(x)
    if not tokens:
        tokens = ["<unk>"]
    return vocab(tokens)

def label_pipeline(y: int):
    # HF ag_news labels are already 0..3
    return int(y)


# ----------------------------
# Collate function for DataLoader (padding + lengths)
# ----------------------------
def collate_batch(batch):
    labels = []
    sequences = []
    lengths = []

    for (label, text) in batch:
        labels.append(label_pipeline(label))
        ids = torch.tensor(text_pipeline(text), dtype=torch.long)
        sequences.append(ids)
        lengths.append(ids.size(0))

    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    tokens_padded = pad_sequence(sequences, batch_first=True, padding_value=pad_idx)
    return tokens_padded, lengths, labels


batch_size = 64
trainloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch, num_workers=0)
testloader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch, num_workers=0)


c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  return _bootstrap._gcd_import(name[level:], package, level)


In [3]:
import torch

In [2]:
# Complete BiRNN example for AG_NEWS (4-class text classification)
# Goal: reproduce the CNN/CIFAR-10 pattern (Dataset -> DataLoader -> Net -> train -> eval) for RNN

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


# ----------------------------
# Model definition (BiRNN)
# __init__: layer configuration
# forward:  data flow
# ----------------------------
class BiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_layers=1, num_classes=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # bidirectional=True -> two directions (forward/backward)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        # combine forward + backward as in the textbook: hidden_size -> num_classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        """
        tokens:  (N, T)  padded token IDs
        lengths: (N,)    actual length of each sentence (before padding)
        return:  (N, num_classes) logits
        """
        # 1. sort by length (for pack) -- enforce_sorted=False also works but explicit here
        lengths_sorted, perm_idx = lengths.sort(0, descending=True)
        tokens_sorted = tokens[perm_idx]

        # 2. Embedding
        x = self.embedding(tokens_sorted)  # (N, T, embed_dim)

        # 3. pack to skip padding in RNN computation
        packed = pack_padded_sequence(x, lengths_sorted.cpu(), batch_first=True)

        # 4. RNN
        # h_n: (num_layers * num_directions, N, hidden_size)
        _, h_n = self.rnn(packed)

        # 5. restore original order (h_n is in sorted order)
        _, unperm_idx = perm_idx.sort(0)
        h_n = h_n[:, unperm_idx, :]

        # 6. combine forward/backward hidden states of the last layer for classification
        # for num_layers>=1: last layer forward is at -2, backward at -1
        h_fwd = h_n[-2]  # (N, hidden_size)
        h_bwd = h_n[-1]  # (N, hidden_size)
        h = h_fwd + h_bwd  # (N, hidden_size)  -- use concat and change fc input dim if preferred

        logits = self.fc(h)  # (N, num_classes)
        return logits


# ----------------------------
# Preprocessing: tokenize -> vocab -> IDs
# ----------------------------
def yield_tokens(data_list, tokenizer):
    for label, text in data_list:
        yield tokenizer(text)


def main(epochs=2, lr=0.05, momentum=0.9, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # tokenizer
    tokenizer = get_tokenizer("basic_english")

    # Dataset (downloaded automatically like CIFAR-10 with download=True)
    train_list = list(AG_NEWS(split="train"))
    test_list = list(AG_NEWS(split="test"))

    # vocab
    special_tokens = ["<unk>", "<pad>"]
    vocab = build_vocab_from_iterator(
        yield_tokens(train_list, tokenizer),
        specials=special_tokens,
        min_freq=2,
    )
    vocab.set_default_index(vocab["<unk>"])
    pad_idx = vocab["<pad>"]

    # pipeline
    def text_pipeline(text: str):
        return vocab(tokenizer(text))

    def label_pipeline(label: int):
        return int(label) - 1  # AG_NEWS labels are 1..4, convert to 0..3

    # collate function for DataLoader (padding + lengths)
    def collate_batch(batch):
        labels = []
        seqs = []
        lengths = []

        for (label, text) in batch:
            labels.append(label_pipeline(label))
            ids = torch.tensor(text_pipeline(text), dtype=torch.long)
            seqs.append(ids)
            lengths.append(ids.size(0))

        labels = torch.tensor(labels, dtype=torch.long)
        lengths = torch.tensor(lengths, dtype=torch.long)

        tokens_padded = pad_sequence(seqs, batch_first=True, padding_value=pad_idx)  # (N, T)
        return tokens_padded, lengths, labels

    trainloader = DataLoader(train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    testloader = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

    # Model / Loss / Optimizer
    net = BiRNN(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)

    # ----------------------------
    # Training
    # ----------------------------
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)     # (N, 4)
            loss = criterion(outputs, labels)  # labels: (N,)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}")
                running_loss = 0.0

    print("Finished Training")

    # ----------------------------
    # Evaluation
    # ----------------------------
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")
    return net


if __name__ == "__main__":
    main()


c:\Users\Nutzer\anaconda3\Lib\site-packages\torchtext\__init__.py:7: SyntaxWarning: invalid escape sequence '\ '
  "\n/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ \n"


OSError: Could not load this library: C:\Users\Nutzer\anaconda3\Lib\site-packages\torchtext\lib\libtorchtext.pyd

In [ ]:
class NetBiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            embed_dim, hidden_size, num_layers=1, batch_first=True,
            bidirectional=True, nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # 2H input because bidirectional

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        # h_n: (layers*dirs, N, H) = (2, N, H); last two are forward and backward
        forward_last = h_n[-2]                # (N, H)
        backward_last = h_n[-1]               # (N, H)
        last_hidden = torch.cat([forward_last, backward_last], dim=1)  # (N, 2H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


## Unidirectional RNN model

```
tokens (N, T)
  → Embedding (N, T, E)
  → pack_padded_sequence
  → nn.RNN  → h_n (1, N, H)  ← hidden state at the final step
  → Linear(H, C)
  → logits (N, C)
```

- `pack_padded_sequence` skips padding positions
- `h_n[-1]` = last layer, last step hidden state (N, H)


In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence


class NetRNN(nn.Module):
    """Unidirectional RNN model for text classification"""

    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        # 1. token ID -> dense vector (gradients at pad positions are ignored)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # 2. unidirectional RNN (bidirectional=False by default)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,       # input shape: (N, T, E)
            nonlinearity="tanh",    # default: tanh
        )

        # 3. hidden state -> number of classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        """
        tokens:  (N, T)  padded token IDs
        lengths: (N,)    actual length of each sentence
        return:  (N, num_classes) logits
        """
        # 1. Embedding
        x = self.embedding(tokens)  # (N, T, embed_dim)

        # 2. convert to PackedSequence to skip padding
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)

        # 3. pass through RNN
        # output: PackedSequence (hidden states at all steps)
        # h_n:    (num_layers * num_directions, N, hidden_size) = (1, N, H)
        _, h_n = self.rnn(packed)

        # 4. extract the last layer, last step hidden state
        last_hidden = h_n[-1]  # (N, hidden_size)

        # 5. classify with fully connected layer
        logits = self.fc(last_hidden)  # (N, num_classes)
        return logits


In [ ]:
import torch.optim as optim

# ----------------------------
# Setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

net = NetRNN(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)

# ----------------------------
# Training loop
# ----------------------------
epochs = 3

for epoch in range(epochs):
    net.train()
    running_loss = 0.0

    for i, (tokens, lengths, labels) in enumerate(trainloader, 1):
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        optimizer.zero_grad()
        outputs = net(tokens, lengths)      # (N, 4)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 0:
            print(f"[epoch {epoch+1}, step {i:5d}]  loss: {running_loss/200:.3f}")
            running_loss = 0.0

print("Finished Training")

# ----------------------------
# Evaluation
# ----------------------------
net.eval()
correct = total = 0

with torch.no_grad():
    for tokens, lengths, labels in testloader:
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        outputs = net(tokens, lengths)
        preds   = outputs.argmax(dim=1)
        total   += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"Test Accuracy: {100.0 * correct / total:.2f}%")


## Bidirectional RNN (BiRNN)

```
tokens (N, T)
  → Embedding (N, T, E)
  → nn.RNN(bidirectional=True)
       forward  →  h_fwd (N, H)   ← reads from start to end
       backward →  h_bwd (N, H)   ← reads from end to start
  → cat([h_fwd, h_bwd], dim=1)  (N, 2H)
  → Linear(2H, C)
  → logits (N, C)
```

- `h_n` shape: `(num_layers * 2, N, H)`
- last layer forward = `h_n[-2]`, backward = `h_n[-1]`


In [ ]:
# NetBiRNN is defined in the cell above
# This cell runs training and evaluation only

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net_birnn = NetBiRNN(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.SGD(net_birnn.parameters(), lr=0.05, momentum=0.9)

# ----------------------------
# Training loop
# ----------------------------
epochs = 3

for epoch in range(epochs):
    net_birnn.train()
    running_loss = 0.0

    for i, (tokens, lengths, labels) in enumerate(trainloader, 1):
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        optimizer.zero_grad()
        outputs = net_birnn(tokens, lengths)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 0:
            print(f"[epoch {epoch+1}, step {i:5d}]  loss: {running_loss/200:.3f}")
            running_loss = 0.0

print("Finished Training")

# ----------------------------
# Evaluation
# ----------------------------
net_birnn.eval()
correct = total = 0

with torch.no_grad():
    for tokens, lengths, labels in testloader:
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        outputs = net_birnn(tokens, lengths)
        preds   = outputs.argmax(dim=1)
        total   += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"Test Accuracy: {100.0 * correct / total:.2f}%")


## LSTM

Unlike RNN, LSTM maintains a **cell state `c`** (long-term memory) in addition to the hidden state `h`.

```
Four gates computed at each step:
  forget gate  f = σ(W_f·[h, x] + b_f)   ← how much of old memory to erase
  input gate   i = σ(W_i·[h, x] + b_i)   ← how much new info to write
  cell gate    g = tanh(W_g·[h, x] + b_g) ← candidate new values
  output gate  o = σ(W_o·[h, x] + b_o)   ← what to output

Cell state update:   c_t = f ⊙ c_{t-1} + i ⊙ g
Hidden state update: h_t = o ⊙ tanh(c_t)
```

In PyTorch, just replace `nn.RNN` with `nn.LSTM` — the calling convention is identical.  
The only difference is the return value: `(output, (h_n, c_n))` instead of `(output, h_n)`.


In [ ]:
class NetLSTM(nn.Module):
    """Unidirectional LSTM model for text classification"""

    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # just swap nn.RNN -> nn.LSTM (same arguments)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)  # (N, T, E)

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)

        # LSTM returns (output, (h_n, c_n))
        # h_n: (num_layers, N, H)  <- hidden state (short-term memory)
        # c_n: (num_layers, N, H)  <- cell state   (long-term memory)
        _, (h_n, _) = self.lstm(packed)

        last_hidden = h_n[-1]          # (N, H)
        logits = self.fc(last_hidden)  # (N, C)
        return logits


# ----------------------------
# Training + Evaluation
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net_lstm  = NetLSTM(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net_lstm.parameters(), lr=0.05, momentum=0.9)

epochs = 3

for epoch in range(epochs):
    net_lstm.train()
    running_loss = 0.0

    for i, (tokens, lengths, labels) in enumerate(trainloader, 1):
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        optimizer.zero_grad()
        outputs = net_lstm(tokens, lengths)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 0:
            print(f"[epoch {epoch+1}, step {i:5d}]  loss: {running_loss/200:.3f}")
            running_loss = 0.0

print("Finished Training")

net_lstm.eval()
correct = total = 0

with torch.no_grad():
    for tokens, lengths, labels in testloader:
        tokens  = tokens.to(device)
        lengths = lengths.to(device)
        labels  = labels.to(device)

        outputs = net_lstm(tokens, lengths)
        preds   = outputs.argmax(dim=1)
        total   += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"Test Accuracy: {100.0 * correct / total:.2f}%")
